# SEED-VII EEGNet Encoder Pipeline

**版本：2026-05-28**

---

## 累积更改一览

| 版本 | 更改内容 |
|------|----------|
| 原始版 | 全被试混合，trial-level 分割，7 类情绪，EEGNet 双头 |
| 重构版 | per-subject .npz，memmap OOM-safe，被试内独立训练入口 |
| 标签聚合 | 7 类 → **3 类**（正面/中性/负面），npz 不重新预处理，加载时 remap |
| 样本均衡 | 训练集 **WeightedRandomSampler**（3 类天然不均衡：正面2种、中性1种、负面4种），val/test 保持原始分布 |
| 训练模式 | **`--training-mode`** 三选一，替代旧的 `--freeze-intensity-head` 布尔开关 |

---

## 训练模式说明（`--training-mode`）

| 模式 | 强度头 | 损失函数 | optimizer 参数 | 日志阶段 | 推荐场景 |
|------|--------|----------|----------------|----------|----------|
| `cls_only` | **冻结** | 全程只有 L_cls | 只含分类参数 | `CLS` | **先跑这个**，任务简单，收敛快 |
| `joint` | 可训练 | 全程 L_cls + L_reg | 全部参数 | `JOINT` | 想同时优化分类和强度回归 |
| `pretrain_then_joint` | 可训练 | 前 N epoch 只 L_cls，之后联合 | 全部参数 | `PRE`→`JOINT` | 渐进式训练 |

## 数据集划分方案

| 方案 | 脚本 | 输出 | 适用场景 |
|------|------|------|----------|
| 全被试混合 | `train.py` | 1 个模型 | 跨被试泛化研究 |
| 每被试独立 | `train_per_subject.py` | 每被试一个模型 | **被试内泛化**（EEG 跨被试泛化差是公认结论） |

**两种方案均严格 trial-level 分割**：同一 trial 的所有时间窗口只出现在 train/val/test 之一，无数据泄漏。

---

## Notebook 结构

```
Cell 0  安装依赖
Cell 1  路径配置
Cell 2  预处理 .mat → .npz
Cell 3  上传 .npz 到 ModelScope
Cell 4  （可选）从 ModelScope 下载 .npz
Cell 5  标签分布检查（3 类聚合后的分布）
──── 训练 ────────────────────────────────────────────────
Cell 6  全被试混合 × cls_only      ← 推荐首次运行
Cell 7  全被试混合 × joint
Cell 8  全被试混合 × pretrain_then_joint
Cell 9  全被试混合 × 续训
Cell 10 每被试独立 × cls_only      ← 被试内泛化推荐
Cell 11 每被试独立 × joint
Cell 12 每被试独立 × pretrain_then_joint
──── 结果 ────────────────────────────────────────────────
Cell 13 编码器推理（全被试模型）
Cell 14 编码器推理（单被试模型，逐被试）
Cell 15 结果汇总 & 对比
```

## Cell 0：安装依赖

In [ ]:
!pip install -q numpy scipy torch tqdm h5py pyyaml modelscope

## Cell 1：路径配置

**修改下方路径变量后再运行后续所有 Cell。**

In [ ]:
import os, sys, json
import numpy as np
from pathlib import Path

# ── 项目根目录（含 src/ scripts/ 等）──────────────────────────
PROJECT_ROOT = Path("/mnt/workspace/EEG_encoder_version2")   # ← 修改

# ── 数据 ──────────────────────────────────────────────────────
MAT_DIR       = Path("/mnt/workspace/data/EEG_preprocessed") # ← 修改
SAVE_INFO_DIR = Path("/mnt/workspace/data/save_info")        # 无则留空
NPZ_DIR       = Path("/mnt/workspace/preprocessed")          # .npz 存放目录

# ── 训练输出目录 ───────────────────────────────────────────────
# 全被试混合训练（按 training_mode 分子目录）
RUNS_ALL = Path("/mnt/workspace/runs_all")
RUNS_ALL_CLS     = RUNS_ALL / "cls_only"
RUNS_ALL_JOINT   = RUNS_ALL / "joint"
RUNS_ALL_PRETRAIN= RUNS_ALL / "pretrain_then_joint"

# 单被试独立训练（按 training_mode 分子目录）
RUNS_SUBJ = Path("/mnt/workspace/runs_per_subject")
RUNS_SUBJ_CLS      = RUNS_SUBJ / "cls_only"
RUNS_SUBJ_JOINT    = RUNS_SUBJ / "joint"
RUNS_SUBJ_PRETRAIN = RUNS_SUBJ / "pretrain_then_joint"

# ── 编码输出 ───────────────────────────────────────────────────
ENCODED_ALL  = Path("/mnt/workspace/encoded_all.npz")
ENCODED_SUBJ = Path("/mnt/workspace/encoded_per_subject")

# ── ModelScope ────────────────────────────────────────────────
MODELSCOPE_DATASET = "DEREKVERSE/SEED-VII"
NPZ_REPO_PREFIX    = "preprocessed_npz"
# os.environ["MODELSCOPE_API_TOKEN"] = "your_token_here"

# ── sys.path ──────────────────────────────────────────────────
sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT : {PROJECT_ROOT} (exists={PROJECT_ROOT.exists()})")
print(f"MAT_DIR      : {MAT_DIR} (exists={MAT_DIR.exists()})")
print(f"NPZ_DIR      : {NPZ_DIR}")
print(f"RUNS_ALL     : {RUNS_ALL}")
print(f"RUNS_SUBJ    : {RUNS_SUBJ}")

## Cell 2：预处理 — 20 个 .mat → 20 个 per-subject .npz

```
原始 EEG (62ch, 200Hz, 已带通+陷波)
  → 基线校正 → CAR → 居中60%裁剪
  → 4s 窗口 50% 重叠（每 trial ≤60 窗口）
  → per-channel z-score
  → {subject}.npz  （y 存储原始 7 类，加载时自动 remap 为 3 类）
```

> **仅首次运行**。后续只需下载/使用已有 .npz。

In [ ]:
!python {PROJECT_ROOT}/scripts/preprocess_to_npz.py \
  --mat-dir {MAT_DIR} \
  --output-dir {NPZ_DIR} \
  --save-info-dir {SAVE_INFO_DIR if SAVE_INFO_DIR.exists() else ''} \
  --window-seconds 4.0 \
  --step-seconds 2.0 \
  --middle-ratio 0.6 \
  --max-windows-per-trial 60 \
  --compress

In [ ]:
# 验证预处理结果（显示原始 7 类分布，训练时会自动 remap 为 3 类）
for p in sorted(NPZ_DIR.glob("*.npz")):
    d = np.load(p, allow_pickle=True)
    uniq, cnt = np.unique(d["y"], return_counts=True)
    print(f"{p.name}: X={d['X'].shape}, 7-class dist={dict(zip(uniq.tolist(), cnt.tolist()))}")

## Cell 3：上传 .npz 到 ModelScope

In [ ]:
!python {PROJECT_ROOT}/scripts/upload_npz_to_modelscope.py \
  --npz-dir {NPZ_DIR} \
  --dataset {MODELSCOPE_DATASET} \
  --path-prefix {NPZ_REPO_PREFIX}

## Cell 4：（可选）从 ModelScope 下载 .npz

In [ ]:
# !python {PROJECT_ROOT}/scripts/download_npz_from_modelscope.py \
#   --dataset {MODELSCOPE_DATASET} \
#   --path-prefix {NPZ_REPO_PREFIX} \
#   --local-dir {NPZ_DIR}

## Cell 5：标签分布检查

确认 3 类聚合后的分布，以及样本均衡的效果。

**标签映射规则**（npz 文件中存原始 7 类，训练时 `scan_npz_metadata(remap_labels=True)` 自动转换）：

| 原始 7 类 | 情绪 | → 3 类 |
|-----------|------|--------|
| H(0), U(1) | Happy, Surprise | **0 = Positive** |
| N(2) | Neutral | **1 = Neutral** |
| D(3), F(4), S(5), A(6) | Disgust, Fear, Sad, Anger | **2 = Negative** |

In [ ]:
from src.labels import remap_labels_3class, VALENCE_LABELS
from src.dataset import scan_npz_metadata

# 轻量扫描（只读 y/s/meta，X 不加载）
npz_paths, y_all, s_all, meta, file_map = scan_npz_metadata(
    str(NPZ_DIR), remap_labels=True   # ← 返回已 remap 的 3 类标签
)

classes, counts = np.unique(y_all, return_counts=True)
total = len(y_all)

print(f"总窗口数：{total}")
print("\n3 类分布（训练时实际使用的标签）：")
for c, cnt in zip(classes, counts):
    print(f"  {VALENCE_LABELS[int(c)]:10s} (cls {c}): {cnt:6d} 窗口  ({cnt/total*100:.1f}%)")

print("\n说明：负面类包含 4 种情绪，天然样本更多。")
print("训练时 WeightedRandomSampler 会将三类采样频率均衡至 1:1:1。")
print("验证集/测试集保持上述原始分布，反映真实场景。")

---

## ─── 训练部分 ───────────────────────────────────────────────────

### 公共配置说明

所有训练命令共享以下设计：
- **标签**：3 类聚合（正面/中性/负面），加载时自动 remap
- **均衡**：训练集 WeightedRandomSampler 默认开启（`--no-balance-train` 关闭）
- **分割**：trial-level，同一 trial 的窗口不跨 split
- **早停**：`--patience 30`，监控 val acc

## Cell 6：全被试混合 × `cls_only`（推荐首次运行）

- 强度头参数**永久冻结**，optimizer 只更新分类相关参数
- 损失函数全程只有 `L_cls`（交叉熵 + 标签平滑），`beta=0`
- 任务简单（3 类），收敛快，过拟合风险低
- 日志阶段标签：`CLS`

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_ALL_CLS} \
  --training-mode cls_only \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

## Cell 7：全被试混合 × `joint`

- 强度头**可训练**，损失全程为 `L_cls + L_reg`
- 适合分类已收敛后想进一步优化强度回归的场景
- 日志阶段标签：`JOINT`

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_ALL_JOINT} \
  --training-mode joint \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --beta-reg 0.5 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

## Cell 8：全被试混合 × `pretrain_then_joint`

- 前 `pretrain-epochs`（默认 10）epoch 只开 `L_cls`（日志：`PRE`）
- 之后切换为 `L_cls + L_reg` 联合训练（日志：`JOINT`）
- 强度头参数**全程可训练**，只是损失的 beta 在预训练阶段为 0

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_ALL_PRETRAIN} \
  --training-mode pretrain_then_joint \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --beta-reg 0.5 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

## Cell 9：全被试混合 × 续训

训练中断后从断点继续，`--training-mode` 需与原始一致。

In [ ]:
# 修改 RESUME_MODE 和 RESUME_DIR 后取消注释运行
RESUME_MODE = "cls_only"           # cls_only / joint / pretrain_then_joint
RESUME_DIR  = RUNS_ALL_CLS         # 对应模式的输出目录

# !python {PROJECT_ROOT}/scripts/train.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RESUME_DIR} \
#   --training-mode {RESUME_MODE} \
#   --model-type eegnet \
#   --device auto --amp \
#   --resume \
#   --max-runtime-hours 10
print("取消注释上方代码后运行。")

---

## Cell 10：每被试独立 × `cls_only`（被试内泛化推荐）

对每个被试（1~20）独立执行：
1. 只加载该被试自己的 `{S}.npz`
2. 在该被试的 trial 内做 trial-level 分割（train 80% / val 10% / test 10%）
3. 从头初始化模型，完整跑 `cls_only` 训练循环
4. 输出到 `runs_per_subject/cls_only/subject_{S}/`
5. 释放显存，进入下一个被试

**输出结构：**
```
runs_per_subject/cls_only/
  subject_1/  best_encoder.pt  summary.json  train.log
  subject_2/  ...
  ...
  all_summary.json  ← 20 被试 test acc 均值 ± 标准差
```

In [ ]:
!python {PROJECT_ROOT}/scripts/train_per_subject.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_SUBJ_CLS} \
  --training-mode cls_only \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 64 \
  --lr 2e-4 \
  --max-epochs 200 \
  --patience 30 \
  --val-ratio 0.1 \
  --test-ratio 0.1 \
  --max-runtime-hours 20

In [ ]:
# 调试：只跑部分被试
# !python {PROJECT_ROOT}/scripts/train_per_subject.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RUNS_SUBJ_CLS} \
#   --training-mode cls_only \
#   --subjects 1,2,3 \
#   --device auto --amp \
#   --max-epochs 50 --patience 15 \
#   --max-runtime-hours 2
print("调试时取消注释上方代码。")

## Cell 11：每被试独立 × `joint`

In [ ]:
!python {PROJECT_ROOT}/scripts/train_per_subject.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_SUBJ_JOINT} \
  --training-mode joint \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 64 \
  --lr 2e-4 \
  --beta-reg 0.5 \
  --max-epochs 200 \
  --patience 30 \
  --val-ratio 0.1 --test-ratio 0.1 \
  --max-runtime-hours 20

## Cell 12：每被试独立 × `pretrain_then_joint`

In [ ]:
!python {PROJECT_ROOT}/scripts/train_per_subject.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_SUBJ_PRETRAIN} \
  --training-mode pretrain_then_joint \
  --model-type eegnet \
  --device auto --amp \
  --batch-size 64 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --beta-reg 0.5 \
  --max-epochs 200 \
  --patience 30 \
  --val-ratio 0.1 --test-ratio 0.1 \
  --max-runtime-hours 20

---

## Cell 13：编码器推理 — 全被试混合模型

In [ ]:
# 选择要推理的训练模式目录
INFER_RUNS_ALL = RUNS_ALL_CLS   # 修改为 RUNS_ALL_JOINT / RUNS_ALL_PRETRAIN

!python {PROJECT_ROOT}/scripts/encode.py \
  --data-dir {NPZ_DIR} \
  --checkpoint {INFER_RUNS_ALL}/best_encoder.pt \
  --output {ENCODED_ALL} \
  --model-type eegnet \
  --device auto

if ENCODED_ALL.exists():
    d   = np.load(str(ENCODED_ALL), allow_pickle=True)
    acc = (d["cls_pred"] == d["labels"]).mean()
    print(f"Features : {d['features'].shape}")
    print(f"Overall acc (3-class): {acc:.4f}")

## Cell 14：编码器推理 — 每被试独立模型（逐被试）

In [ ]:
import subprocess

# 选择要推理的训练模式目录
INFER_RUNS_SUBJ = RUNS_SUBJ_CLS   # 修改为 RUNS_SUBJ_JOINT / RUNS_SUBJ_PRETRAIN

ENCODED_SUBJ.mkdir(exist_ok=True)

for subj_dir in sorted(INFER_RUNS_SUBJ.glob("subject_*")):
    subj = subj_dir.name.replace("subject_", "")
    ckpt = subj_dir / "best_encoder.pt"
    if not ckpt.exists():
        print(f"[Subject {subj}] best_encoder.pt 不存在，跳过")
        continue
    out = ENCODED_SUBJ / f"encoded_{subj}.npz"
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "encode.py"),
        "--data-dir",   str(NPZ_DIR),
        "--checkpoint", str(ckpt),
        "--output",     str(out),
        "--subjects",   subj,
        "--model-type", "eegnet",
        "--device",     "auto",
    ]
    subprocess.run(cmd, check=True)
    d   = np.load(str(out), allow_pickle=True)
    acc = (d["cls_pred"] == d["labels"]).mean()
    print(f"[Subject {subj}] features={d['features'].shape}  acc={acc:.4f}")

## Cell 15：结果汇总 & 对比

对比不同训练模式的结果。

In [ ]:
# ── 全被试混合模型：各模式 summary ───────────────────────────
print("=" * 60)
print("全被试混合训练结果")
print("=" * 60)
for mode, runs_dir in [
    ("cls_only",           RUNS_ALL_CLS),
    ("joint",              RUNS_ALL_JOINT),
    ("pretrain_then_joint",RUNS_ALL_PRETRAIN),
]:
    sp = runs_dir / "summary.json"
    if sp.exists():
        s = json.load(open(sp))
        ta = s.get("test", {}).get("acc", float("nan"))
        print(f"  {mode:<24} val_acc={s['best_val_acc']:.4f}  "
              f"test_acc={ta:.4f}  epochs={s['epochs_run']}")
    else:
        print(f"  {mode:<24} （尚未训练）")

# ── 单被试模型：各模式汇总 ────────────────────────────────────
print()
print("=" * 60)
print("单被试独立训练结果")
print("=" * 60)
for mode, runs_dir in [
    ("cls_only",           RUNS_SUBJ_CLS),
    ("joint",              RUNS_SUBJ_JOINT),
    ("pretrain_then_joint",RUNS_SUBJ_PRETRAIN),
]:
    ap = runs_dir / "all_summary.json"
    if ap.exists():
        a = json.load(open(ap))
        print(f"  {mode:<24} n={a['n_subjects']}  "
              f"val_acc={a['mean_val_acc']:.4f}±{a['std_val_acc']:.4f}  "
              f"test_acc={a['mean_test_acc']:.4f}±{a['std_test_acc']:.4f}")
    else:
        print(f"  {mode:<24} （尚未训练）")

In [ ]:
# ── 单被试 cls_only 各被试明细 ────────────────────────────────
ap = RUNS_SUBJ_CLS / "all_summary.json"
if ap.exists():
    a = json.load(open(ap))
    print(f"单被试 cls_only 各被试明细（训练模式：{a.get('training_mode','?')}）")
    print(f"  {'Subject':>10} | {'Val Acc':>8} | {'Test Acc':>9} | {'Test MAE':>9} | {'Epochs':>6}")
    print("  " + "-" * 55)
    for subj, sm in sorted(a["per_subject"].items()):
        ta  = sm.get("test", {}).get("acc",           float("nan"))
        mae = sm.get("test", {}).get("intensity_mae", float("nan"))
        print(f"  {subj:>10} | {sm['best_val_acc']:>8.4f} | "
              f"{ta:>9.4f} | {mae:>9.4f} | {sm['epochs_run']:>6}")
else:
    print("all_summary.json 不存在，请先运行 Cell 10。")

In [ ]:
# ── 查看某个被试的训练日志（尾部 30 行）──────────────────────
SUBJECT_TO_INSPECT = "1"          # ← 修改
LOG_MODE           = "cls_only"   # ← 修改：cls_only / joint / pretrain_then_joint

mode_dir_map = {
    "cls_only":            RUNS_SUBJ_CLS,
    "joint":               RUNS_SUBJ_JOINT,
    "pretrain_then_joint": RUNS_SUBJ_PRETRAIN,
}
log_path = mode_dir_map[LOG_MODE] / f"subject_{SUBJECT_TO_INSPECT}" / "train.log"

if log_path.exists():
    lines = open(log_path).readlines()
    print(f"--- Subject {SUBJECT_TO_INSPECT} [{LOG_MODE}] 最后 30 行 ---")
    for line in lines[-30:]:
        print(line, end="")
else:
    print(f"{log_path} 不存在")